<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/PAPSS2_Top50_Old.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PAPSS2 — OLD DERIVATIVES
# GLOBAL TOP 10 FROM THE FINAL TOP 50
#
# OUTPUT:
# 1. Global_Top10_Old_Derivatives.sdf
# 2. Global_Top10_Old_Derivatives_Methodology.docx
# 3. Global_Top10_Old_Derivatives_Table.docx
# ============================================================

!pip -q install rdkit openpyxl python-docx

import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdDepictor
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from google.colab import files

In [2]:
# ------------------------------------------------------------
# STEP 1 — UPLOAD FINAL TOP 50 EXCEL FILE
# ------------------------------------------------------------

uploaded = files.upload()

excel_file = next(
    (f for f in uploaded if f.lower().endswith(".xlsx")),
    None
)

if not excel_file:
    raise ValueError("Upload PAPSS2_Final_Top10_Per_Library.xlsx")

df = pd.read_excel(excel_file)


Saving PAPSS2_Top50_Old_docked.xlsx to PAPSS2_Top50_Old_docked.xlsx


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# ------------------------------------------------------------
# STEP 2 — IDENTIFY REQUIRED COLUMNS
# ------------------------------------------------------------

def find_column(possible_names, columns):
    normalized = {
        str(c).strip().lower().replace(" ", "_"): c
        for c in columns
    }

    for name in possible_names:
        key = name.lower().strip().replace(" ", "_")
        if key in normalized:
            return normalized[key]

    return None

cid_col = find_column(
    ["PUBCHEM COMPOUND CID", "PUBCHEM_CID", "CID"],
    df.columns
)

name_col = find_column(
    ["PUBCHEM IUPAC NAME", "COMPOUND_NAME", "IUPAC_NAME"],
    df.columns
)

edock_col = find_column(
    ["Edock", "E_dock"],
    df.columns
)

smiles_col = find_column(
    ["Canonical_Isomeric_SMILES",
     "CANONICAL_ISOMERIC_SMILES",
     "Canonical Isomeric SMILES"],
    df.columns
)

library_col = find_column(
    ["Library", "LIBRARY"],
    df.columns
)

if not all([cid_col, name_col, edock_col, smiles_col]):
    raise ValueError(
        "Required columns not found. Available columns:\n"
        + str(df.columns.tolist())
    )


In [4]:
# ------------------------------------------------------------
# STEP 3 — GLOBAL RANKING OF ALL 50 COMPOUNDS
# ------------------------------------------------------------

df[edock_col] = pd.to_numeric(
    df[edock_col], errors="coerce"
)

df = df.dropna(
    subset=[cid_col, edock_col, smiles_col]
).copy()

df[cid_col] = df[cid_col].astype(str).str.strip()

# Keep the best Edock record for each CID
df = (
    df.sort_values(edock_col, ascending=True)
      .drop_duplicates(subset=[cid_col], keep="first")
)

# Global top 10 — NOT 10 per library
top10 = df.sort_values(
    edock_col, ascending=True
).head(10).copy()

top10.insert(0, "Global_Rank", range(1, len(top10) + 1))

if len(top10) != 10:
    raise ValueError("Could not identify 10 valid compounds.")

print("GLOBAL TOP 10 SELECTED")
print(top10[[cid_col, name_col, edock_col]].to_string(index=False))

GLOBAL TOP 10 SELECTED
PUBCHEM CID                                                                                                                                                          COMPOUND NAME     EDOCK
      64148                                                                                  2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyclo[3.3.1.13,7]decane 2,2,6,6-tetraoxide -84.31694
   44263409                                                                      4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phosphonooxypropan-2-yl]carbamoyl]piperidine-2-carboxylate -83.90470
   11181229                                                            1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl-3-phosphonooxypropan-2-yl]carbamoyl]piperidine-2-carboxylate -80.23081
     455458                                                                                                4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperidin-1-yl)propyl]piperidine -74.13148
    5281148                     

In [5]:
# ------------------------------------------------------------
# STEP 4 — GENERATE ONE SDF FILE
# ------------------------------------------------------------

sdf_file = "Global_Top10_Old_Derivatives.sdf"

writer = Chem.SDWriter(sdf_file)

for _, row in top10.iterrows():

    smiles = str(row[smiles_col]).strip()
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            f"Invalid SMILES for CID {row[cid_col]}"
        )

    mol = Chem.AddHs(mol)
    rdDepictor.Compute2DCoords(mol)

    mol.SetProp("_Name", f"CID_{row[cid_col]}")
    mol.SetProp("PUBCHEM_CID", str(row[cid_col]))
    mol.SetProp("IUPAC_NAME", str(row[name_col]))
    mol.SetProp("GLOBAL_RANK", str(row["Global_Rank"]))
    mol.SetProp("EDOCK", str(row[edock_col]))
    mol.SetProp("CANONICAL_ISOMERIC_SMILES", smiles)
    mol.SetProp(
        "STRUCTURE_NOTE",
        "2D coordinates generated from SMILES; not an ICM-Pro docked pose."
    )

    if library_col:
        mol.SetProp("SOURCE_LIBRARY", str(row[library_col]))

    # Preserve other available docking parameters
    for col in ["Egv", "Egb", "Ege", "Egs",
                "LE_like", "RDKit_Heavy_Atoms"]:
        actual_col = find_column([col], top10.columns)
        if actual_col and pd.notna(row[actual_col]):
            mol.SetProp(col.upper(), str(row[actual_col]))

    writer.write(mol)

writer.close()

# Validate SDF
supplier = Chem.SDMolSupplier(sdf_file, removeHs=False)
valid_molecules = [m for m in supplier if m is not None]

if len(valid_molecules) != 10:
    raise ValueError("SDF validation failed.")

print("\nSDF validation:", len(valid_molecules), "valid compounds")



SDF validation: 10 valid compounds


In [6]:
# ------------------------------------------------------------
# STEP 4 — GENERATE ONE SDF FILE
# ------------------------------------------------------------

sdf_file = "Global_Top10_Old_Derivatives.sdf"

writer = Chem.SDWriter(sdf_file)

for _, row in top10.iterrows():

    smiles = str(row[smiles_col]).strip()
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            f"Invalid SMILES for CID {row[cid_col]}"
        )

    mol = Chem.AddHs(mol)
    rdDepictor.Compute2DCoords(mol)

    mol.SetProp("_Name", f"CID_{row[cid_col]}")
    mol.SetProp("PUBCHEM_CID", str(row[cid_col]))
    mol.SetProp("IUPAC_NAME", str(row[name_col]))
    mol.SetProp("GLOBAL_RANK", str(row["Global_Rank"]))
    mol.SetProp("EDOCK", str(row[edock_col]))
    mol.SetProp("CANONICAL_ISOMERIC_SMILES", smiles)
    mol.SetProp(
        "STRUCTURE_NOTE",
        "2D coordinates generated from SMILES; not an ICM-Pro docked pose."
    )

    if library_col:
        mol.SetProp("SOURCE_LIBRARY", str(row[library_col]))

    # Preserve other available docking parameters
    for col in ["Egv", "Egb", "Ege", "Egs",
                "LE_like", "RDKit_Heavy_Atoms"]:
        actual_col = find_column([col], top10.columns)
        if actual_col and pd.notna(row[actual_col]):
            mol.SetProp(col.upper(), str(row[actual_col]))

    writer.write(mol)

writer.close()

# Validate SDF
supplier = Chem.SDMolSupplier(sdf_file, removeHs=False)
valid_molecules = [m for m in supplier if m is not None]

if len(valid_molecules) != 10:
    raise ValueError("SDF validation failed.")

print("\nSDF validation:", len(valid_molecules), "valid compounds")



SDF validation: 10 valid compounds


In [7]:
# ------------------------------------------------------------
# STEP 6 — CREATE METHODOLOGY WORD FILE
# ------------------------------------------------------------

method_file = "Global_Top10_Old_Derivatives_Methodology.docx"

method = Document()

method.add_heading(
    "PAPSS2 — Old Derivatives: Global Top 10 Screening Methodology",
    level=1
)

method.add_heading("1. Objective", level=2)
method.add_paragraph(
    "To identify the 10 highest-ranked compounds from the final "
    "50-compound PAPSS2 shortlist using a single global ranking "
    "based on Edock scores."
)

method.add_heading("2. Input dataset", level=2)
method.add_paragraph(
    "The input was the final shortlist containing up to 50 compounds "
    "selected previously as the top 10 from each of five chemical "
    "libraries: phytochemicals, adamantane derivatives, "
    "benzodiazepine derivatives, piperidines and pyrrolidines."
)

method.add_heading("3. Global selection procedure", level=2)
steps = [
    "Import the final 50-compound Excel workbook.",
    "Convert Edock values to numeric format.",
    "Remove records with missing CID, Edock or canonical isomeric SMILES.",
    "Remove duplicate PubChem CIDs, retaining the record with the "
    "most negative Edock score.",
    "Sort all remaining compounds together by ascending Edock score.",
    "Select the first 10 compounds as the global top 10."
]
for step in steps:
    method.add_paragraph(step, style="List Number")

method.add_heading("4. SDF structure preparation", level=2)
method.add_paragraph(
    "Canonical isomeric SMILES were parsed using RDKit. Valid molecular "
    "structures were hydrogenated and 2D coordinates were generated. "
    "The resulting structures were written into one SDF file, with "
    "compound identifiers, global ranks and Edock values stored as "
    "molecule properties."
)

method.add_heading("5. Validation", level=2)
method.add_paragraph(
    f"The generated SDF was reopened and checked using RDKit. "
    f"Valid records obtained: {len(valid_molecules)}. "
    "Each record represents one selected compound."
)

method.add_heading("6. Interpretation and limitations", level=2)
method.add_paragraph(
    "The global ranking is based on Edock scores only. Other docking "
    "energy components and molecular descriptors are retained as "
    "supporting information where available; they were not combined "
    "into an invented composite score."
)
method.add_paragraph(
    "The generated SDF contains 2D structures derived from SMILES. "
    "It does not contain original ICM-Pro docked poses, protein-bound "
    "conformations or residue-level interaction information."
)
method.add_paragraph(
    "The exact docking protocol settings should be documented separately "
    "from the original ICM-Pro records or laboratory notes."
)

method.add_heading("7. Final output", level=2)
method.add_paragraph(
    "One global top-10 compound table, one SDF containing 10 molecular "
    "records, and this methodology document."
)

method.save(method_file)

In [11]:
from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files
import pandas as pd
import os

# Check that top10 still exists
if "top10" not in globals():
    raise ValueError("The top10 variable is missing. Rerun the selection cell.")

table_file = "Global_Top10_Old_Derivatives_Table.docx"

doc = Document()

section = doc.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width, section.page_height = (
    section.page_height, section.page_width
)
section.left_margin = Inches(0.4)
section.right_margin = Inches(0.4)

doc.add_heading(
    "PAPSS2 — Global Top 10 Old Derivatives",
    level=1
)

p = doc.add_paragraph(
    "Global ranking of the final 50 compounds based on ascending "
    "Edock scores. The most negative Edock score ranks first."
)
p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

# Include available columns
preferred = [
    "Global_Rank",
    cid_col,
    name_col,
    library_col,
    edock_col,
    "Egv", "Egb", "Ege", "Egs",
    "LE_like",
    "RDKit_Heavy_Atoms"
]

columns = []
for col in preferred:
    if col and col in top10.columns and col not in columns:
        columns.append(col)

table = doc.add_table(rows=1, cols=len(columns))
table.style = "Table Grid"

for i, col in enumerate(columns):
    table.rows[0].cells[i].text = str(col)

for _, row in top10.iterrows():
    cells = table.add_row().cells
    for i, col in enumerate(columns):
        value = row[col]
        cells[i].text = "" if pd.isna(value) else str(value)

doc.add_paragraph(
    "Note: The SDF contains generated 2D molecular structures, "
    "not original ICM-Pro docked poses."
)

doc.save(table_file)

print("Word table created:", table_file)

# Download all three files
for filename in [
    "Global_Top10_Old_Derivatives.sdf",
    "Global_Top10_Old_Derivatives_Methodology.docx",
    table_file
]:
    if os.path.exists(filename):
        files.download(filename)
    else:
        print("Missing file:", filename)

Word table created: Global_Top10_Old_Derivatives_Table.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files
from rdkit import Chem
from rdkit.Chem import rdDepictor
import pandas as pd
import os

# Check existing selection
if "top10" not in globals() or len(top10) != 10:
    raise ValueError("The global top10 selection is unavailable. Rerun the selection cell.")

# --------------------------------------------------
# 1. CREATE TOP 10 SDF
# --------------------------------------------------

sdf_file = "Global_Top10_Old_Derivatives.sdf"
writer = Chem.SDWriter(sdf_file)

for _, row in top10.iterrows():
    mol = Chem.MolFromSmiles(str(row[smiles_col]))
    if mol is None:
        raise ValueError(f"Invalid structure for CID {row[cid_col]}")

    mol = Chem.AddHs(mol)
    rdDepictor.Compute2DCoords(mol)

    mol.SetProp("_Name", f"CID_{row[cid_col]}")
    mol.SetProp("PUBCHEM_CID", str(row[cid_col]))
    mol.SetProp("IUPAC_NAME", str(row[name_col]))
    mol.SetProp("GLOBAL_RANK", str(row["Global_Rank"]))
    mol.SetProp("EDOCK", str(row[edock_col]))
    mol.SetProp("STRUCTURE_NOTE", "Generated 2D structure; not an ICM-Pro docked pose.")

    if library_col:
        mol.SetProp("SOURCE_LIBRARY", str(row[library_col]))

    writer.write(mol)

writer.close()

# Validate
valid = [m for m in Chem.SDMolSupplier(sdf_file, removeHs=False) if m]
assert len(valid) == 10, "SDF validation failed."

# --------------------------------------------------
# 2. CREATE RESULTS TABLE WORD FILE
# --------------------------------------------------

table_file = "Global_Top10_Old_Derivatives_Table.docx"

doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = sec.page_height, sec.page_width
sec.left_margin = Inches(0.4)
sec.right_margin = Inches(0.4)

doc.add_heading("PAPSS2 — Global Top 10 Old Derivatives", 0)
doc.add_paragraph(
    "Ten compounds selected globally from the final 50-compound shortlist "
    "by ascending Edock score. More negative scores rank higher."
)

columns = [
    c for c in [
        "Global_Rank", cid_col, name_col, library_col, edock_col,
        "Egv", "Egb", "Ege", "Egs", "LE_like", "RDKit_Heavy_Atoms"
    ] if c and c in top10.columns
]

tbl = doc.add_table(rows=1, cols=len(columns))
tbl.style = "Table Grid"

for i, col in enumerate(columns):
    tbl.rows[0].cells[i].text = str(col)

for _, row in top10.iterrows():
    cells = tbl.add_row().cells
    for i, col in enumerate(columns):
        value = row[col]
        cells[i].text = "" if pd.isna(value) else str(value)

doc.add_heading("Screening limitations", level=2)
doc.add_paragraph(
    "Binding interactions and binding poses were not assessed because "
    "original ICM-Pro docked poses were unavailable. The SDF contains "
    "generated 2D structures, not docked conformations."
)
doc.save(table_file)

# --------------------------------------------------
# 3. CREATE METHODOLOGY WORD FILE
# --------------------------------------------------

method_file = "Global_Top10_Old_Derivatives_Methodology.docx"

method = Document()
method.add_heading(
    "PAPSS2 — Old Derivatives: Global Top 10 Methodology", 0
)

sections = [
    ("Objective",
     "To prioritise ten compounds globally from the final 50-compound "
     "PAPSS2 shortlist."),
    ("Primary criterion",
     "Edock score. All eligible compounds were ranked in ascending order; "
     "more negative values were considered more favourable."),
    ("Supporting criteria",
     "Egv, Egb, Ege and Egs were reviewed for additional scoring information "
     "and unusual values. Ligand efficiency was assessed using the internal "
     "LE-like metric. Chemical diversity was evaluated using molecular "
     "fingerprint similarity and maximum common substructure analysis."),
    ("Mandatory structural check",
     "Molecular structures were retrieved using PubChem CIDs and validated "
     "using RDKit. All 50 shortlisted structures were successfully parsed."),
    ("Selection procedure",
     "Duplicate PubChem CIDs were reduced to their best Edock record. "
     "The remaining compounds were ranked together, and the first ten "
     "were selected as one global shortlist."),
    ("Binding interactions and pose",
     "Hydrogen bonds, hydrophobic contacts, salt bridges, residue-level "
     "interactions and docked-pose plausibility were not assessed because "
     "the original ICM-Pro docked poses were unavailable."),
    ("Limitations",
     "The selection is a preliminary computational prioritisation, not a "
     "fully pose-validated shortlist. The SDF contains 2D structures "
     "generated from canonical SMILES, not original ICM-Pro docked poses. "
     "No arbitrary weighted scoring formula was used.")
]

for heading, paragraph in sections:
    method.add_heading(heading, level=2)
    method.add_paragraph(paragraph)

method.add_heading("Final output", level=2)
method.add_paragraph(
    "One global top-10 shortlist from 50 compounds, one SDF containing "
    "ten molecular records, and one results table."
)

method.save(method_file)

# --------------------------------------------------
# 4. CHECK AND DOWNLOAD ALL THREE
# --------------------------------------------------

for f in [sdf_file, method_file, table_file]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing output: {f}")
    print(f"READY: {f} | {os.path.getsize(f):,} bytes")

print("\nAll three files are ready.")

files.download(sdf_file)
files.download(method_file)
files.download(table_file)

READY: Global_Top10_Old_Derivatives.sdf | 58,827 bytes
READY: Global_Top10_Old_Derivatives_Methodology.docx | 37,407 bytes
READY: Global_Top10_Old_Derivatives_Table.docx | 38,084 bytes

All three files are ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>